# Pairs Trading with Mean-Reverting Stochastic Processes
**MAC F244 project (Topic 17)**: Ornstein-Uhlenbeck spreads, ADF/Engle-Granger pair selection and optimal trading bands, on US and Indian stocks (2010-2026).

**How to use this notebook (temporary master copy)**
- Every section is tagged with its owner (M1-M5). Edit only your own sections, then tick your task in `Project_Tracker.csv`.
- The maths and the backtest live in `src/`; this notebook calls them and makes every figure and table in the report.
- Figures are saved to `figs/` as PNG and EPS with `savefig(fig, name)`, using the names `report/main.tex` expects.
- Cells marked **HEAVY** run the full walk-forward backtests (slow the first time, cached in `results/cache/` afterwards). They only run when `RUN_HEAVY = True`.
- Before submission, M1 runs *Restart & Run All* once and commits the outputs.

| Section | Content | Owner |
| --- | --- | --- |
| 1 | Setup | M1 |
| 2 | Data | M2 |
| 3 | OU model and simulation | M3 |
| 4 | E1 estimator accuracy, E2a ADF power, E2b test size | M3, M4, M2 |
| 5 | Pair selection on one window | M2 |
| 6 | Trading rules; Bertram bands and E3 | M4; M5 |
| 7 | Backtest engine checks, E4 synthetic pair | M1 |
| 8 | E5 headline backtest | M1, M4 |
| 9 | E6 transaction costs | M3 |
| 10 | E7 parameter sensitivity | M3 |
| 11 | E8 markets, sectors, sub-periods | M3, M2, M5 |
| 12 | E9 cointegration persistence | M2 |
| 13 | E10 Kalman hedge ratio (stretch) | M5 |
| 14 | Conclusions | M3 |

## 1. Setup  (Owner: M1)
Imports, the fixed seed (42) and the switch for the heavy backtests. All parameters live in `src/config.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_prices, benchmark_returns, sectors
from src.ou import (simulate_ou, fit_ou, bertram_cycle_time, bertram_bands,
                    bertram_return_rate, first_passage_cycle_mc)
from src.cointegration import adf_test, engle_granger, engle_granger_both, naive_residual_adf
from src.selection import select_cointegrated, select_distance
from src.backtest import run_selection, simulate, pair_returns
from src.metrics import compute_metrics, metrics_table
from src.plots import savefig, COLORS

np.random.seed(C.SEED)
rng = np.random.default_rng(C.SEED)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

RUN_HEAVY = False   # True runs the full walk-forward backtests (sections 8-13)
STRATS = ['distance', 'zscore', 'sscore', 'bertram']

## 2. Data  (Owner: M2)
Daily adjusted closes from Yahoo Finance (`yfinance`), cached in `data/raw/` (see `data/raw/README.md` for retrieval date and flagged jumps). 56 US stocks in 7 sectors plus EWA/EWC, and 26 NSE stocks in 5 sectors. Pairs are formed only within a sector.

*M2: add your 5-line notes on Gatev et al. (2006) and Engle & Granger (1987) here (task T04).*

In [ ]:
px_us = load_prices('us')
px_in = load_prices('india')
bench_us = benchmark_returns('us')
bench_in = benchmark_returns('india')

summary = pd.DataFrame({
    m: {'tickers': p.shape[1], 'first day': p.index[0].date(), 'last day': p.index[-1].date(),
        'trading days': len(p), 'missing %': round(100 * p.isna().mean().mean(), 2)}
    for m, p in {'US': px_us, 'India': px_in}.items()}).T
summary

In [ ]:
# Two classic pairs, prices normalised to 1 at the start
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('XOM', 'CVX')]):
    p = px_us[[a, b]].dropna()
    (p / p.iloc[0]).plot(ax=ax, lw=1)
    ax.set_title(f'{a} vs {b}'); ax.set_ylabel('Normalised price'); ax.set_xlabel('')
savefig(fig, 'fig_data_pairs')

## 3. The Ornstein-Uhlenbeck model  (Owner: M3)
The log-price spread $X_t = \ln P^A_t - \beta \ln P^B_t - \mu$ is modelled as
$$dX_t = \kappa(\theta - X_t)\,dt + \sigma\,dW_t, \qquad \kappa > 0.$$
Itô's lemma on $e^{\kappa t}X_t$ gives
$$X_t = \theta + (X_0-\theta)e^{-\kappa t} + \sigma\int_0^t e^{-\kappa(t-s)}\,dW_s,$$
so $X_t$ is Gaussian with mean $\theta + (X_0-\theta)e^{-\kappa t}$, variance $\sigma^2(1-e^{-2\kappa t})/(2\kappa)$, stationary law $N(\theta, \sigma^2/2\kappa)$ and half-life $\ln 2/\kappa$.

**Exact discretisation.** With step $\Delta = 1/252$:
$$X_{t+\Delta} = a + bX_t + \varepsilon_t,\quad b = e^{-\kappa\Delta},\quad a = \theta(1-b),\quad \mathrm{Var}(\varepsilon_t) = \sigma^2\frac{1-e^{-2\kappa\Delta}}{2\kappa}.$$
OLS of $X_{t+\Delta}$ on $X_t$ is the conditional MLE, and $\hat\kappa = -\ln\hat b/\Delta$, $\hat\theta = \hat a/(1-\hat b)$, $\sigma_{eq} = \hat s_\varepsilon/\sqrt{1-\hat b^2}$.

**Euler-Maruyama** uses $X_{t+\Delta} = X_t + \kappa(\theta - X_t)\Delta + \sigma\sqrt{\Delta}Z$, i.e. $b \approx 1-\kappa\Delta$. The noise is additive, so Milstein's correction term is zero and Milstein equals Euler-Maruyama.

*M3: check every line against report section 2 (task T05).*

In [ ]:
# Exact vs Euler-Maruyama paths driven by the same shocks (same seed)
kappa, theta, sigma, n = 25.0, 0.0, 0.2, 252
x_exact = simulate_ou(kappa, theta, sigma, 0.08, n, method='exact', rng=np.random.default_rng(1))[0]
x_euler = simulate_ou(kappa, theta, sigma, 0.08, n, method='euler', rng=np.random.default_rng(1))[0]
sig_eq = sigma / np.sqrt(2 * kappa)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x_exact, lw=1, label='Exact scheme')
ax.plot(x_euler, lw=1, ls='--', label='Euler-Maruyama')
ax.axhspan(theta - sig_eq, theta + sig_eq, color='grey', alpha=0.15, label=r'$\theta \pm \sigma_{eq}$')
ax.set_xlabel('Trading day'); ax.set_ylabel('Spread'); ax.legend()
savefig(fig, 'fig_ou_paths')

# T11: the estimator recovers known parameters on a long exact path
for k in [5, 25, 50]:
    x = simulate_ou(k, 0.0, 0.2, 'stationary', 252 * 40, rng=rng)[0]
    p = fit_ou(x)
    print(f'kappa={k:>3}: kappa_hat={p.kappa:6.2f}, theta_hat={p.theta:+.4f}, sigma_hat={p.sigma:.4f}, half-life={p.half_life:5.1f} d')

## 4. E1 and E2: can we estimate and detect mean reversion?  (Owners: M3 E1, M4 E2a, M2 E2b)
**E1.** Monte Carlo of $\hat\kappa$ for windows of 63-504 days. Yu (2012) shows $\hat\kappa$ is biased upward in short samples, which makes half-lives look shorter than they are. We also compare the exact estimator with the Euler estimator $(1-\hat b)/\Delta$.

**E2.** The ADF regression $\Delta X_t = \alpha + \gamma X_{t-1} + \sum_i\delta_i\Delta X_{t-i} + \varepsilon_t$ has $\gamma = e^{-\kappa\Delta}-1$, so $H_0:\gamma=0$ is $H_0:\kappa=0$. We measure (a) the power of ADF across half-life and window length, which justifies the 12-month formation window, and (b) the size of the residual test on two independent random walks: plain ADF critical values vs Engle-Granger (MacKinnon 2010) critical values (spurious regression, Granger & Newbold 1974).

In [ ]:
# E1: bias and RMSE of kappa_hat
kappas, lengths, n_paths = [5, 10, 25, 50], [63, 126, 252, 504], 1000
rows = []
for k in kappas:
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=n_paths, rng=rng)
        k_exact = np.array([fit_ou(p).kappa for p in x])
        k_euler = np.array([fit_ou(p, estimator='euler').kappa for p in x])
        rows.append(dict(kappa=k, days=T, valid_share=np.isfinite(k_exact).mean(),
                         rel_bias_exact=np.nanmean(k_exact) / k - 1,
                         rel_rmse_exact=np.sqrt(np.nanmean((k_exact - k) ** 2)) / k,
                         rel_bias_euler=np.nanmean(k_euler) / k - 1))
e1 = pd.DataFrame(rows)
e1.to_csv('results/e1_ou_estimation.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 3.6))
for k in kappas:
    d = e1[e1.kappa == k]
    ax.plot(d.days, 100 * d.rel_bias_exact, marker='o', label=f'kappa = {k} / yr')
ax.set_xscale('log'); ax.set_xticks(lengths); ax.set_xticklabels(lengths)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('Relative bias of kappa_hat (%)'); ax.legend()
savefig(fig, 'fig_e1_ou_estimation')
e1

In [ ]:
# E2a: ADF power over half-life x window length (rejection rate at 5%)
half_lives, lengths, reps = [5, 10, 20, 40, 80], [63, 126, 252, 504], 300
power = pd.DataFrame(index=half_lives, columns=lengths, dtype=float)
for hl in half_lives:
    k = np.log(2) / (hl * C.DT)
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=reps, rng=rng)
        power.loc[hl, T] = np.mean([adf_test(p, maxlag=5)[1] < 0.05 for p in x])
power.to_csv('results/e2_adf_power.csv')

fig, ax = plt.subplots(figsize=(5.5, 3.6))
im = ax.imshow(power.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(lengths)), lengths); ax.set_yticks(range(len(half_lives)), half_lives)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('True half-life (days)')
for i in range(len(half_lives)):
    for j in range(len(lengths)):
        ax.text(j, i, f'{power.values[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
fig.colorbar(im, label='ADF rejection rate')
ax.grid(False)
savefig(fig, 'fig_e2_adf_power')
power

In [ ]:
# E2b: size of the residual test when the truth is 'not cointegrated'
reps, n = 1000, 252
naive, eg = [], []
for _ in range(reps):
    y, x = np.cumsum(rng.standard_normal((2, n)) * 0.01, axis=1)
    naive.append(naive_residual_adf(y, x)[1] < 0.05)
    eg.append(engle_granger(y, x).pvalue < 0.05)
e2_size = pd.Series({'Plain ADF critical values on residuals': np.mean(naive),
                     'Engle-Granger (MacKinnon) critical values': np.mean(eg)},
                    name='Rejection rate of a nominal 5% test')
e2_size.to_csv('results/e2_size.csv')
e2_size

*M3 (E1), M4 (E2a), M2 (E2b): interpret your experiment here in 2 sentences each (bias size at 252 days, which half-lives a 12-month window can detect, how badly the naive test over-rejects).*

## 5. Pair selection on one formation window  (Owner: M2)
Engle-Granger in both directions (keep a pair only if both p-values are below 5%), positive hedge ratio, half-life 2-63 trading days, at least 12 mean crossings a year (Sarmento & Horta 2020), ranked by the EG statistic, top 20. Benjamini-Hochberg (q = 0.10) is reported as a diagnostic. These numbers go into the proposal's *progress so far* paragraph (task T13).

In [ ]:
form = np.log(px_us.iloc[-C.FORMATION_DAYS:]).dropna(axis=1)
print(f'Formation window: {form.index[0].date()} to {form.index[-1].date()}')
chosen, diag, tested = select_cointegrated(form, sectors('us'))
print(diag)
top = pd.DataFrame([dict(pair=f'{s.a}/{s.b}', sector=s.sector, beta=s.beta, eg_stat=s.eg_stat,
                         eg_p=s.eg_pvalue, half_life_days=s.half_life, crossings_per_year=s.crossings)
                    for s in chosen])
top.head(10)

## 6. Trading rules and Bertram's optimal bands  (Owners: M4 rules, M5 Bertram and E3)
- **S0 distance** (Gatev et al. 2006): open when normalised prices diverge by 2 formation standard deviations, close at the crossing.
- **S1 z-score**: open at $|z|>2$, close at $|z|<0.5$.
- **S2 s-score** (Avellaneda & Lee 2010): $s = (X-\hat\theta)/\sigma_{eq}$; open long $s<-1.25$, open short $s>1.25$, close long $s>-0.50$, close short $s<0.75$.
- **S3 Bertram (2010)**: for the centred OU with entry $a$, exit $m$ and round-trip cost $c$,
$$\mathbb{E}[T] = \frac{\pi}{\kappa}\left[\mathrm{erfi}\left(\frac{m\sqrt\kappa}{\sigma}\right) - \mathrm{erfi}\left(\frac{a\sqrt\kappa}{\sigma}\right)\right],\qquad \mu(a,m) = \frac{m-a-c}{\mathbb{E}[T]},$$
from the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$. With $a=-m$ we maximise $\mu$ numerically.

*M4: 5-line notes on Avellaneda & Lee (task T06). M5: 5-line notes on Bertram (task T49).*

In [ ]:
# Proposal figure (T12): spread, OU fit and s-score bands on the latest year
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('EWA', 'EWC')]):
    lp = np.log(px_us[[a, b]].iloc[-C.FORMATION_DAYS:].dropna())
    best, p_both = engle_granger_both(lp[a].values, lp[b].values, a, b)
    ou = fit_ou(best.resid)
    s = (best.resid - ou.theta) / ou.sigma_eq if ou.valid else best.resid / best.resid.std()
    ax.plot(lp.index, s, lw=1)
    for lvl, ls in [(1.25, '-'), (-1.25, '-'), (0.75, ':'), (-0.5, ':')]:
        ax.axhline(lvl, color='grey', ls=ls, lw=0.8)
    hl = f'{ou.half_life:.0f} d' if ou.valid else 'n/a'
    ax.set_title(f'{best.y} - {best.beta:.2f} {best.x}: EG p = {p_both:.2f}, half-life {hl}')
    ax.set_ylabel('s-score')
savefig(fig, 'fig_proposal_spread')

In [ ]:
# E3: Bertram's E[T] against simulated first-passage cycles, and m* against cost
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)
m_grid = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0]) * sig_eq
formula = bertram_cycle_time(-m_grid, m_grid, kappa, sigma) * 252
mc = [first_passage_cycle_mc(kappa, sigma, m, n_steps=1_000_000, rng=rng).mean() * 252 for m in m_grid]
e3 = pd.DataFrame({'m / sigma_eq': m_grid / sig_eq, 'E[T] formula (days)': formula, 'E[T] Monte Carlo (days)': mc})
e3.to_csv('results/e3_bertram_check.csv', index=False)

costs = np.linspace(0, 1.0, 21) * sig_eq
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(e3.iloc[:, 0], formula, label='Formula')
axes[0].plot(e3.iloc[:, 0], mc, 'o', label='Monte Carlo')
axes[0].set_xlabel('Band half-width m / sigma_eq'); axes[0].set_ylabel('Expected cycle (days)'); axes[0].legend()
for k in [5, 20, 50]:
    sigma_k = sig_eq * np.sqrt(2 * k)   # same sigma_eq, different speed
    m_star = [bertram_bands(k, sigma_k, c) / sig_eq for c in costs]
    axes[1].plot(costs / sig_eq, m_star, label=f'kappa = {k}')
axes[1].set_xlabel('Round-trip cost / sigma_eq'); axes[1].set_ylabel('Optimal m* / sigma_eq'); axes[1].legend()
savefig(fig, 'fig_e3_bertram')
e3

## 7. Backtest engine checks and E4 synthetic pair  (Owner: M1)
**Timing rules (no look-ahead).** Pairs, tickers and every parameter come from the 252-day formation window only. The signal computed at the close of day $t$ is executed at the close of $t+1$ (the one-day wait of Gatev et al. 2006), so day $t$'s return is earned by the position decided at the close of $t-2$. Open positions are closed on the last day of the 126-day trading window.

**P&L.** One unit of spread is \$1 long $A$ and \$$\beta$ short $B$. Since $r^A-\beta r^B \approx d\ln P^A-\beta\,d\ln P^B = dX$, the position earns the change of the OU spread. Daily return on gross capital: $\text{pos}_{t-1}(r^A_t-\beta r^B_t)/(1+|\beta|)$, minus the cost per leg on every unit traded (a flip long$\to$short is two units).

**Checks.** `tests/` holds 20 tests with hand-computable answers. The decisive one scrambles every price after a date (and deletes some) and checks that no earlier return changes.

In [ ]:
# T14, T22-T24: unit tests of the engine and the metrics, then the look-ahead check by hand
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True).stdout.strip().splitlines()[-1])

desired = np.zeros(30); desired[10:20] = 1          # signal on at the close of day 10
ra = np.full(30, 0.01); rb = np.zeros(30)
net, held = pair_returns(desired, 1.0, ra, rb, cost=0.0)
assert held[10] == 0 and held[11] == 1, 'execution lag broken'
assert net[11] == 0 and net[12] > 0, 'return earned before the position was held'
print('Look-ahead check passed: signal at close 10, bought at close 11, first return on day 12')

**Integration check (T25).** One real pair through the whole pipeline on one window: selection on 2018, trading in the first half of 2019.

In [ ]:
from src.backtest import make_windows
w = make_windows(len(px_us))[16]
chosen, diag, _ = select_cointegrated(np.log(px_us.iloc[w.f0:w.t0]), sectors('us'))
spec = chosen[0]
print(f'Formation {px_us.index[w.f0].date()} to {px_us.index[w.t0 - 1].date()}, trading {px_us.index[w.t0].date()} to {px_us.index[w.t1 - 1].date()}')
print(f'{diag["n_eg"]}/{diag["n_tested"]} pairs pass Engle-Granger, {diag["n_filtered"]} pass all filters')
print(f'Top pair {spec.a}/{spec.b} ({spec.sector}): beta = {spec.beta:.3f}, EG p = {spec.eg_pvalue:.4f}, '
      f'kappa = {spec.kappa:.1f}/yr, half-life = {spec.half_life:.1f} d, sigma_eq = {spec.sigma_eq:.4f}')
one = [dict(window=w, coint=[spec], distance=[], diag=diag)]
res_one = simulate('sscore', px_us, one, C.COST_BPS['us'], k=1)

lp = np.log(px_us[[spec.a, spec.b]].iloc[w.f0:w.t1])
s = (lp[spec.a] - spec.beta * lp[spec.b] - spec.mu - spec.theta) / spec.sigma_eq
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(s.index, s.values, lw=0.9, color=COLORS['sscore'])
ax.axvspan(px_us.index[w.t0], px_us.index[w.t1 - 1], color='grey', alpha=0.12)
ax.text(px_us.index[w.t0 + 3], -3.0, 'trading window: parameters frozen', fontsize=8)
ax.set_ylim(-3.3, 2.7)
for lvl in (1.25, -1.25): ax.axhline(lvl, color='grey', lw=0.8)
for lvl in (0.75, -0.5): ax.axhline(lvl, color='grey', lw=0.8, ls=':')
for _, t in res_one.trades.iterrows():
    ax.axvspan(t.open, t.close, ymin=0.92, ymax=1.0, color='k' if t.side > 0 else 'grey')
ax.set_ylabel('s-score'); ax.set_title(f'{spec.a} - {spec.beta:.2f} {spec.b}: bars on top mark trades (black long, grey short)')
savefig(fig, 'fig_m1_integration')
print(compute_metrics(res_one.committed, trades=res_one.trades)['Total return %'], '% on the pair over the window')
res_one.trades

**E4 (T27): does the pipeline earn what the OU theory predicts?** A synthetic pair with a known OU spread ($\kappa = 25$/yr, half-life 7 days, $\sigma_{eq} = 0.03$, $\beta = 1.2$, 5 bp per leg) is pushed through the pipeline 200 times. For this pair Bertram's formula predicts the return of the optimal bands exactly, so we add one real-world friction at a time and measure what each costs:

| Level | What changes |
| --- | --- |
| L0 | Bertram's formula: continuous monitoring, true parameters |
| L1 | spread seen once a day, traded at the same close, true parameters |
| L2 | + one-day execution lag |
| L3 | + 6-month trading windows (start flat, forced close) |
| L4a | + OU parameters $\kappa,\theta,\sigma$ estimated on the formation year (true $\beta$) |
| L4 | + $\beta$ and $\mu$ estimated too |
| L5 | + pair traded only when it passes Engle-Granger and the OU filters (the full pipeline) |

In [ ]:
# E4 ladder: 200 synthetic pairs, about 90 s; cached in results/
import json, os
from src.synthetic import run_ladder
if os.path.exists('results/e4_ladder.csv') and os.path.exists('results/e4_theory.json'):
    e4 = pd.read_csv('results/e4_ladder.csv'); e4_info = json.load(open('results/e4_theory.json'))
else:
    e4, e4_info = run_ladder(reps=200)
    e4.to_csv('results/e4_ladder.csv', index=False)
    json.dump({k: float(v) for k, v in e4_info.items()}, open('results/e4_theory.json', 'w'), indent=1)

levels = ['L1', 'L2', 'L3', 'L4a', 'L4', 'L5']
names = {'L0': 'L0 Bertram formula', 'L1': 'L1 daily monitoring', 'L2': 'L2 + one-day lag', 'L3': 'L3 + 6-month windows',
         'L4a': 'L4a + estimated OU params', 'L4': 'L4 + estimated beta', 'L5': 'L5 + EG selection (pipeline)'}
n = len(e4)
table_e4 = pd.DataFrame({'Ann. return %': [100 * e4_info['theory']] + [100 * e4[l].mean() for l in levels],
                         'SE %': [np.nan] + [100 * e4[l].std() / np.sqrt(n) for l in levels]},
                        index=[names[l] for l in ['L0'] + levels])
steps = [e4['L1'] - e4_info['theory']] + [e4[b] - e4[a] for a, b in zip(levels, levels[1:])]
table_e4['Step %'] = [np.nan] + [100 * d.mean() for d in steps]
table_e4['Step SE %'] = [np.nan, np.nan] + [100 * d.std() / np.sqrt(n) for d in steps[1:]]   # paired SE
table_e4['Share of theory %'] = 100 * table_e4['Ann. return %'] / table_e4['Ann. return %'].iloc[0]
table_e4.to_csv('results/e4_table.csv')
print(f"Fine-grid simulation of L0 (50 steps a day, 200 years): {100 * e4_info['fine_grid']:.1f}% vs formula {100 * e4_info['theory']:.1f}%")
print(f"Pair selected in {100 * e4.selected_share.mean():.0f}% of windows; median kappa_hat / kappa = {e4.kappa_hat_over_true.mean():.2f}; "
      f"median |beta_hat - beta| = {e4.beta_abs_err.mean():.3f}; m*/sigma_eq true {e4_info['m_star']:.3f}, fitted {e4.m_hat_in_sig_eq.mean():.3f}")
print(f"s-score on the same pipeline (L5): {100 * e4.sscore_L5.mean():.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.4))
y = np.arange(len(table_e4))[::-1]
ax.barh(y, table_e4['Ann. return %'], height=0.6, color=COLORS['bertram'], xerr=2 * table_e4['SE %'].fillna(0),
        error_kw=dict(lw=1, capsize=2, ecolor='k'))
ax.axvline(100 * e4_info['theory'], color='k', lw=0.8, ls='--')
for yi, v, se in zip(y, table_e4['Ann. return %'], table_e4['SE %'].fillna(0)):
    ax.text(v + 2 * se + 0.4, yi, f'{v:.1f}%', va='center', fontsize=8, bbox=dict(fc='white', ec='none', pad=0.5))
ax.set_yticks(y, table_e4.index); ax.set_xlabel('Annual return on gross capital (%), mean of 200 pairs, ±2 SE')
ax.set_xlim(0, 100 * e4_info['theory'] * 1.15); ax.grid(axis='y', visible=False)
savefig(fig, 'fig_e4_synthetic')
table_e4.round(2)

**E4 result.** The fine-grid simulation reproduces Bertram's formula (24.3% vs 24.2% a year), so the formula and our P&L bookkeeping agree. The full pipeline earns 12.7% (SE 0.2%), about half the theoretical return, and every step of the gap has a cause:

* **Daily monitoring (−1.1 pp).** The spread can cross a band and come back within a day, so some cycles are missed. Each daily step moves $0.42\,\sigma_{eq}$, almost as much as the band half-width $0.49\,\sigma_{eq}$.
* **One-day lag (−2.2 pp).** By the time we trade, the spread has already reverted by about $1-e^{-\kappa\Delta}\approx 9\%$ of its distance from the mean. This is the price of an honest execution rule.
* **Window resets (−0.5 pp).** Each window starts flat and ends with a forced close.
* **Estimated OU parameters (−1.0 pp).** $\hat\kappa$ is biased upward by about 24% at 252 days (Yu 2012, our E1), but the optimal band in units of $\sigma_{eq}$ barely moves ($0.50$ vs $0.49$), so the loss is small.
* **Estimated hedge ratio (−2.3 pp).** A median error of only $|\hat\beta-\beta| = 0.064$ adds $(\beta-\hat\beta)\ln P^B$ to the traded spread: a random walk of typical size $0.064\times0.015\times\sqrt{126}\approx 0.36\,\sigma_{eq}$ over a trading window, drifting against bands only $0.49\,\sigma_{eq}$ wide. Hedge-ratio error costs more than all OU-parameter errors together (motivates the Kalman hedge, E10).
* **Selection (−4.5 pp).** Engle-Granger at 5% in both directions recognises this truly cointegrated pair in only 74% of formation windows (the power limit measured in E2), and the capital sits idle otherwise: $0.74 \times 17.2\% = 12.7\%$, exactly L5.

The s-score rule on the same pipeline earns 9.7%, below Bertram's 12.7%: on a true OU pair the bands built for maximum return per unit time do win. A single 12-year run is noisy (standard deviation 2.9% across pairs; the first seed we tried gave 8.6%), which is why E4 averages 200 pairs.

## 8. E5 headline backtest, US  (Owners: M1 engine, M4 strategies)  HEAVY
Rolling 12-month formation and 6-month trading windows, K = 20 pairs, 10 bp per leg. Design choices are frozen on 2010-2021; 2022-2026 is the untouched test block.

In [ ]:
if RUN_HEAVY:
    sel_us = run_selection('us', px_us, sectors('us'))     # slow once, then cached
    res_us = {s: simulate(s, px_us, sel_us, C.COST_BPS['us'], label=s) for s in STRATS}
    table1 = metrics_table(res_us.values(), bench_us).rename(index=C.STRATEGY_LABELS)
    table1.to_csv('results/table1_us.csv')
    split = pd.DataFrame({
        (C.STRATEGY_LABELS[s], blk): compute_metrics(r.committed[sl])
        for s, r in res_us.items()
        for blk, sl in [('2010-2021', slice(None, '2021-12-31')), ('2022-2026 test', slice(C.TEST_START, None))]
    }).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    split.to_csv('results/table1_split.csv')
    display(table1.round(2)); display(split.round(2))

In [ ]:
if RUN_HEAVY:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for s, r in res_us.items():
        ax.plot((1 + r.committed).cumprod(), lw=1.2, color=COLORS[s], label=C.STRATEGY_LABELS[s])
    ax.axvspan(pd.Timestamp(C.TEST_START), px_us.index[-1], color='grey', alpha=0.12, label='Test block')
    ax.set_ylabel('Wealth on committed capital (start = 1)'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e5_cumulative')

**Is any difference real? (M1, T40)** A Sharpe ratio estimated from 16 years of data has a standard error of about $1/\sqrt{16} = 0.25$, so differences between rules can easily be noise. We test each OU rule against the distance baseline with a circular block bootstrap of the Sharpe difference (21-day blocks, the same days for both rules; Ledoit & Wolf 2008), and correct the three p-values with Holm's method because we run three tests.

In [ ]:
if RUN_HEAVY:
    from src.metrics import sharpe_diff_bootstrap
    rows = {}
    for s in ['zscore', 'sscore', 'bertram']:
        for blk, sl in [('2011-2026', slice(None)), ('2022-2026 test', slice(C.TEST_START, None))]:
            rows[(C.STRATEGY_LABELS[s], blk)] = sharpe_diff_bootstrap(res_us[s].committed[sl], res_us['distance'].committed[sl])
    sig = pd.DataFrame(rows).T
    for blk in sig.index.get_level_values(1).unique():          # Holm step-down over the three rules
        p = sig.xs(blk, level=1)['p_value'].sort_values()
        adj = np.minimum(1, np.maximum.accumulate(p.values * np.arange(len(p), 0, -1)))
        for name, a in zip(p.index, adj):
            sig.loc[(name, blk), 'p_Holm'] = a
    sig.to_csv('results/significance_vs_distance.csv'); display(sig.round(3))

**How big is look-ahead bias here? (M1)** The same selected pairs, traded three ways: honestly; with execution at the signal's own close (lag 0); and with $\beta$, $\mu$ and the OU parameters re-estimated on the trading window itself, i.e. on prices not yet seen (`peeking_selections`, demonstration only).

In [ ]:
if RUN_HEAVY:
    from src.backtest import peeking_selections
    peek = peeking_selections(sel_us, px_us)
    rows = {}
    for s in STRATS:
        rows[(C.STRATEGY_LABELS[s], 'honest')] = compute_metrics(res_us[s].committed)
        rows[(C.STRATEGY_LABELS[s], 'lag 0')] = compute_metrics(simulate(s, px_us, sel_us, C.COST_BPS['us'], lag=0).committed)
        if s != 'distance':
            rows[(C.STRATEGY_LABELS[s], 'peeking')] = compute_metrics(simulate(s, px_us, peek, C.COST_BPS['us']).committed)
    bias = pd.DataFrame(rows).T[['Ann. return %', 'Sharpe', 'NW t-stat', 'Max DD %']]
    bias.to_csv('results/m1_lookahead_bias.csv'); display(bias.round(2))

*M4: interpret Table 1: which rule wins, is the Sharpe ratio significant (Lo SE, Newey-West t), and is the strategy market-neutral (beta)?*

## 9. E6 transaction costs  (Owner: M3)  HEAVY
Costs enter linearly, so the break-even cost per leg is where the annual return reaches zero.

In [ ]:
if RUN_HEAVY:
    cost_grid = [0, 5, 10, 20, 30]
    e6 = pd.DataFrame([dict(strategy=s, cost_bps=c, **compute_metrics(simulate(s, px_us, sel_us, c).committed))
                       for s in STRATS for c in cost_grid])
    e6.to_csv('results/e6_costs.csv', index=False)
    breakeven = {}
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for s in STRATS:
        d = e6[e6.strategy == s]
        ax.plot(d.cost_bps, d['Sharpe'], marker='o', color=COLORS[s], label=C.STRATEGY_LABELS[s])
        slope, intercept = np.polyfit(d.cost_bps, d['Ann. return %'], 1)
        breakeven[C.STRATEGY_LABELS[s]] = -intercept / slope if slope < 0 else np.nan
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Cost per leg per transaction (bp)'); ax.set_ylabel('Sharpe ratio'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e6_costs')
    display(pd.Series(breakeven, name='Break-even cost (bp per leg)').round(1))

## 10. E7 parameter sensitivity  (Owner: M3)  HEAVY
Reported as analysis only; the headline result keeps the paper thresholds.

In [ ]:
if RUN_HEAVY:
    entries, exits = [1.0, 1.25, 1.5, 2.0, 2.5], [0.0, 0.25, 0.5, 0.75]
    grid = pd.DataFrame(index=entries, columns=exits, dtype=float)
    for e in entries:
        for x in exits:
            p = dict(open_long=-e, open_short=e, close_long=-x, close_short=x)
            grid.loc[e, x] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], params=p).committed)['Sharpe']
    grid.to_csv('results/e7_threshold_grid.csv')
    fig, ax = plt.subplots(figsize=(5, 3.6))
    im = ax.imshow(grid.values.astype(float), cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(exits)), exits); ax.set_yticks(range(len(entries)), entries)
    ax.set_xlabel('Exit |s|'); ax.set_ylabel('Entry |s|'); ax.grid(False)
    fig.colorbar(im, label='Sharpe ratio (S2, 10 bp)')
    savefig(fig, 'fig_e7_heatmap')

    robust = {}
    for f_days in [126, 252, 504]:
        sel_f = run_selection('us', px_us, sectors('us'), formation=f_days)
        robust[f'Formation {f_days} d'] = compute_metrics(simulate('sscore', px_us, sel_f, C.COST_BPS['us']).committed)
    for k in [5, 10, 20]:
        robust[f'K = {k}'] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], k=k).committed)
    robust['Stop-loss |s| > 4'] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], stop=4.0).committed)
    e7 = pd.DataFrame(robust).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    e7.to_csv('results/e7_robustness.csv'); display(e7.round(2))

## 11. E8 markets, sectors and sub-periods  (Owners: M2 India selection, M5 India table, M3 US sector and sub-period tables)  HEAVY

In [ ]:
if RUN_HEAVY:
    sel_in = run_selection('india', px_in, sectors('india'))
    res_in = {s: simulate(s, px_in, sel_in, C.COST_BPS['india'], label=s) for s in STRATS}
    table_in = metrics_table(res_in.values(), bench_in).rename(index=C.STRATEGY_LABELS)
    table_in.to_csv('results/table_india.csv'); display(table_in.round(2))

    # Sharpe of S2 by sector (US): the sub-portfolio of that sector's selected pairs
    r = res_us['sscore']
    by_sector = {}
    for sec, cols in r.pairs.groupby('sector')['pair']:
        by_sector[sec] = compute_metrics(r.pair_returns[list(cols)].sum(axis=1))['Sharpe']
    display(pd.Series(by_sector, name='S2 Sharpe by sector').round(2))

    # Sub-periods, all strategies (US)
    sub = pd.DataFrame({name: {C.STRATEGY_LABELS[s]: compute_metrics(res.committed[a:b])['Sharpe']
                               for s, res in res_us.items()}
                        for name, (a, b) in C.SUB_PERIODS.items()})
    sub.to_csv('results/e8_subperiods.csv'); display(sub.round(2))

## 12. E9 do ADF-selected pairs stay cointegrated?  (Owner: M2)  HEAVY
For every selected pair we re-test the trading-window spread, built with the formation $\beta$, using ADF (no $\beta$ is re-estimated, so plain ADF critical values are correct here).

In [ ]:
if RUN_HEAVY:
    logp_us = np.log(px_us)
    rows = []
    for sel in sel_us:
        w = sel['window']
        for s in sel['coint']:
            x = (logp_us[s.a] - s.beta * logp_us[s.b] - s.mu).iloc[w.t0:w.t1].dropna().values
            if len(x) > 40:
                rows.append(dict(window=w.i, pair=f'w{w.i:02d}:{s.a}/{s.b}', eg_stat=s.eg_stat,
                                 half_life=s.half_life, still_coint=adf_test(x, maxlag=5)[1] < 0.05))
    e9 = pd.DataFrame(rows).merge(res_us['sscore'].pairs[['pair', 'pnl']], on='pair')
    print(f'Share of selected pairs still cointegrated out of sample: {100 * e9.still_coint.mean():.1f}%')
    e9['decile'] = pd.qcut(e9.eg_stat, 10, labels=False) + 1
    by_dec = e9.groupby('decile').pnl.mean() * 100
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.bar(by_dec.index, by_dec.values, color=COLORS['sscore'])
    ax.set_xlabel('Formation EG statistic decile (1 = most negative)'); ax.set_ylabel('Mean pair P&L per window (%)')
    savefig(fig, 'fig_e9_persistence')
    e9.to_csv('results/e9_persistence.csv', index=False)
    display(e9.groupby('still_coint').pnl.describe())

## 13. E10 Kalman-filter hedge ratio (stretch)  (Owner: M5)  HEAVY
State $(\mu_t,\beta_t)$ follows a random walk, observation $\ln P^A_t = \mu_t + \beta_t\ln P^B_t + e_t$ (Elliott et al. 2005; Chan 2013). Trade the standardised forecast error at $\pm 1$.

In [ ]:
if RUN_HEAVY:
    res_us['kalman'] = simulate('kalman', px_us, sel_us, C.COST_BPS['us'], label='kalman')
    display(metrics_table([res_us['kalman']], bench_us).rename(index=C.STRATEGY_LABELS).round(2))

## 14. Conclusions  (Owner: M3)
*Answer the three research questions in one short paragraph each, using the numbers above:*
1. How reliably do ADF and Engle-Granger find mean-reverting spreads (E2, E9)?
2. How well does the OU model describe them, and how accurate are its estimates (E1, E3, E4)?
3. Do OU-based rules beat the distance rule after costs, across markets and periods (E5-E8)?

*Limitations: survivorship bias (today's large caps), fixed hedge ratio within a window, daily closes only.*